# Analyse du catalogue Netflix
8PRO408 — Projet de session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'scipy'

## 1. Chargement et nettoyage

In [ ]:
# le fichier a des lignes mal formatees dans la colonne Overview
# on utilise engine='python' + on_bad_lines='skip' pour les ignorer
df = pd.read_csv("data/netflix.csv", engine="python", on_bad_lines="skip")
print(df.shape)
df.head()

In [ ]:
print(df.dtypes)
print()
print(df.isnull().sum())

In [ ]:
# Vote_Count et Vote_Average sont lus en string a cause des lignes corrompues
df["Vote_Count"] = pd.to_numeric(df["Vote_Count"], errors="coerce")
df["Vote_Average"] = pd.to_numeric(df["Vote_Average"], errors="coerce")

# convertir la date et extraire l'annee
df["Release_Date"] = pd.to_datetime(df["Release_Date"], errors="coerce")
df["year"] = df["Release_Date"].dt.year

# extraire le genre principal (le premier de la liste)
df["main_genre"] = df["Genre"].str.split(",").str[0].str.strip()

# supprimer les lignes sans note ni votes (inutilisables pour la regression)
df = df.dropna(subset=["Vote_Average", "Vote_Count", "Popularity"])
print(f"apres nettoyage : {df.shape}")
df.dtypes

In [ ]:
df.describe()

## 2. Exploration

In [ ]:
# nombre de films par annee
films_par_an = df.groupby("year").size()
fig, ax = plt.subplots(figsize=(10, 4))
films_par_an.plot(kind="bar", ax=ax)
ax.set_title("nombre de contenus par annee")
ax.set_ylabel("nombre")
plt.tight_layout()
plt.show()

In [ ]:
# top 10 genres
top_genres = df["main_genre"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 4))
top_genres.plot(kind="barh", ax=ax)
ax.set_title("top 10 genres")
plt.tight_layout()
plt.show()

In [ ]:
# top 10 langues
top_lang = df["Original_Language"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 4))
top_lang.plot(kind="barh", ax=ax)
ax.set_title("top 10 langues")
plt.tight_layout()
plt.show()

In [ ]:
# distribution des notes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["Vote_Average"].hist(bins=30, ax=axes[0])
axes[0].set_title("distribution Vote_Average")
axes[0].set_xlabel("note")

df["Popularity"].hist(bins=50, ax=axes[1])
axes[1].set_title("distribution Popularity")
axes[1].set_xlabel("popularite")

df["Vote_Count"].hist(bins=50, ax=axes[2])
axes[2].set_title("distribution Vote_Count")
axes[2].set_xlabel("nombre de votes")

plt.tight_layout()
plt.show()

In [ ]:
# note moyenne par genre (top 10)
genre_stats = df.groupby("main_genre").agg(
    note_moy=("Vote_Average", "mean"),
    nb_films=("Title", "count")
).query("nb_films >= 30").sort_values("note_moy", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
genre_stats["note_moy"].head(15).plot(kind="barh", ax=ax)
ax.set_xlabel("note moyenne")
ax.set_title("note moyenne par genre (min 30 films)")
plt.tight_layout()
plt.show()

In [ ]:
# evolution de la note moyenne par annee
note_par_an = df.groupby("year")["Vote_Average"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
note_par_an.plot(ax=ax, marker="o")
ax.set_title("note moyenne par annee")
ax.set_ylabel("Vote_Average")
ax.set_xlabel("annee")
plt.tight_layout()
plt.show()

## 3. Correlations

In [ ]:
# matrice de correlation sur les colonnes numeriques
num_cols = ["Popularity", "Vote_Count", "Vote_Average", "year"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha="right")
ax.set_yticklabels(num_cols)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=11)
plt.colorbar(im)
ax.set_title("matrice de correlation")
plt.tight_layout()
plt.show()

In [ ]:
# scatter : Vote_Count vs Vote_Average
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["Vote_Count"], df["Vote_Average"], alpha=0.2, s=5)
ax.set_xlabel("nombre de votes")
ax.set_ylabel("note moyenne")
ax.set_title("Vote_Count vs Vote_Average")
plt.tight_layout()
plt.show()

r = df[["Vote_Count", "Vote_Average"]].corr().iloc[0,1]
print(f"correlation : {r:.4f}")

In [ ]:
# scatter : Popularity vs Vote_Average
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["Popularity"], df["Vote_Average"], alpha=0.2, s=5)
ax.set_xlabel("popularite")
ax.set_ylabel("note moyenne")
ax.set_title("Popularity vs Vote_Average")
plt.tight_layout()
plt.show()

r = df[["Popularity", "Vote_Average"]].corr().iloc[0,1]
print(f"correlation : {r:.4f}")

In [ ]:
# scatter : annee vs note
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["year"], df["Vote_Average"], alpha=0.15, s=5)
ax.set_xlabel("annee")
ax.set_ylabel("note moyenne")
ax.set_title("annee vs Vote_Average")
plt.tight_layout()
plt.show()

r = df[["year", "Vote_Average"]].dropna().corr().iloc[0,1]
print(f"correlation : {r:.4f}")

## 4. Regression

In [ ]:
# regression lineaire : Vote_Count -> Vote_Average
clean = df.dropna(subset=["Vote_Count", "Vote_Average"])
x = clean["Vote_Count"].values
y = clean["Vote_Average"].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
print(f"Vote_Count -> Vote_Average")
print(f"pente={slope:.6f}, intercept={intercept:.4f}")
print(f"R2={r_value**2:.4f}, p={p_value:.2e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.15, s=5)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color="red", linewidth=2)
ax.set_xlabel("nombre de votes")
ax.set_ylabel("note moyenne")
ax.set_title(f"regression Vote_Count -> Vote_Average (R2={r_value**2:.4f})")
plt.tight_layout()
plt.show()

In [ ]:
# regression : Popularity -> Vote_Average
clean = df.dropna(subset=["Popularity", "Vote_Average"])
x = clean["Popularity"].values
y = clean["Vote_Average"].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
print(f"Popularity -> Vote_Average")
print(f"pente={slope:.6f}, intercept={intercept:.4f}")
print(f"R2={r_value**2:.4f}, p={p_value:.2e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.15, s=5)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color="red", linewidth=2)
ax.set_xlabel("popularite")
ax.set_ylabel("note moyenne")
ax.set_title(f"regression Popularity -> Vote_Average (R2={r_value**2:.4f})")
plt.tight_layout()
plt.show()

In [ ]:
# regression : year -> Vote_Average
clean = df.dropna(subset=["year", "Vote_Average"])
x = clean["year"].values
y = clean["Vote_Average"].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
print(f"year -> Vote_Average")
print(f"pente={slope:.6f}, intercept={intercept:.4f}")
print(f"R2={r_value**2:.4f}, p={p_value:.2e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.15, s=5)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color="red", linewidth=2)
ax.set_xlabel("annee")
ax.set_ylabel("note moyenne")
ax.set_title(f"regression year -> Vote_Average (R2={r_value**2:.4f})")
plt.tight_layout()
plt.show()

In [ ]:
# regression : Vote_Count -> Popularity
clean = df.dropna(subset=["Vote_Count", "Popularity"])
x = clean["Vote_Count"].values
y = clean["Popularity"].values

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
print(f"Vote_Count -> Popularity")
print(f"pente={slope:.6f}, intercept={intercept:.4f}")
print(f"R2={r_value**2:.4f}, p={p_value:.2e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.15, s=5)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, color="red", linewidth=2)
ax.set_xlabel("nombre de votes")
ax.set_ylabel("popularite")
ax.set_title(f"regression Vote_Count -> Popularity (R2={r_value**2:.4f})")
plt.tight_layout()
plt.show()

### Resume des regressions

In [ ]:
# tableau recapitulatif
regressions = [
    ("Vote_Count", "Vote_Average"),
    ("Popularity", "Vote_Average"),
    ("year", "Vote_Average"),
    ("Vote_Count", "Popularity"),
]

results = []
for x_col, y_col in regressions:
    c = df.dropna(subset=[x_col, y_col])
    slope, intercept, r_val, p_val, _ = stats.linregress(c[x_col], c[y_col])
    results.append({
        "x": x_col,
        "y": y_col,
        "pente": round(slope, 6),
        "R2": round(r_val**2, 4),
        "p_value": f"{p_val:.2e}"
    })

pd.DataFrame(results)

## Conclusion

A completer avec l'interpretation des resultats.